# 01_07_build_amortization_drp

Полный пересчёт амортизации терминалов (Jan–Jul 2026) и перезапись
`sandbox_ai.shestopalov_terminal_amortization_model`.

## Порядок
1. **Логика** — lake → CDWH → Excel → `price/48` + окно 48м + exclude брендов/моделей.
2. **Проверка до загрузки** — дубли, месяцы, суммы, exclude (на локальном `amort_df`).
3. **Загрузка** — DROP/CREATE/ORC в Impala.
4. **Smoke после загрузки** — месяцы в озере + сверка суммы с локальным расчётом.

Полная копия со старыми QC: `01_07_build_amortization_drp_FULL_QC_BACKUP.ipynb`.


## 1. Обновлённая логика расчёта

Активные терминалы (lake) → модели (CDWH) → цены (`term_model.xlsx`) →
амортизация `price/48` в окне 48 месяцев с `first_d_ter_delivery` →
обнуление для excluded брендов/моделей.


In [ ]:
import re
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

# Параметры периода: Jan-Aug 2026 (граница периода: 2026-09-01)
period_start = '2026-01-01'
period_end_exclusive = '2026-09-01'
period_months = pd.date_range(period_start, pd.to_datetime(period_end_exclusive) - pd.Timedelta(days=1), freq='MS')

term_model_excel_path = '/home/jovyan/documents/Equaring/Data/term_model.xlsx'
source_file = 'term_model.xlsx'

target_table = 'sandbox_ai.shestopalov_terminal_amortization_model'
save_table_orc_name = 'shestopalov_terminal_amortization_model_2026_01_2026_08.orc'

# CDWH credentials (as agreed earlier)
khd_user = 'DS_LII'
khd_password = 'dl3S$wolx5dz'

print('period_months =', [m.strftime('%Y-%m') for m in period_months])
print('term_model_excel_path =', term_model_excel_path)
print('target_table =', target_table)


def clean_keys(values):
    out = []
    for v in values:
        s = str(v).strip()
        if s and s not in {'None', 'nan', 'NaN'}:
            out.append(s)
    return sorted(set(out))


def sql_in(values):
    vals = clean_keys(values)
    if not vals:
        return "''"
    return ', '.join(["'" + x.replace("'", "''") + "'" for x in vals])


def norm_model(v):
    if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
        return None
    s = str(v).strip().lower().replace('\xa0', ' ')
    if s in {'none', 'nan', 'null', '<null>', 'nat'}:
        return None
    s = s.replace('ё', 'е')
    s = re.sub(r'\s+', ' ', s)
    return s if s else None


def norm_terminal_key(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    if not s:
        return None
    if re.fullmatch(r'\d+', s):
        return str(int(s))
    return s.upper()




def clean_cdwh_text(v):
    """Normalize CDWH text: real nulls stay NaN (never the string 'None'/'nan')."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() in {"none", "nan", "null", "<null>", "nat"}:
        return np.nan
    return s


def is_blank_series(s):
    """True where value is null / empty / literal None|nan|null."""
    def _blank(v):
        if v is None or (isinstance(v, float) and pd.isna(v)) or pd.isna(v):
            return True
        t = str(v).strip()
        return t == "" or t.lower() in {"none", "nan", "null", "<null>", "nat"}

    return s.map(_blank)

def iter_chunks(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def split_nter_for_id_and_code(values):
    id_numeric = []
    code_values = []
    for v in values:
        s = str(v).strip()
        if not s:
            continue
        code_values.append(s)
        if re.fullmatch(r'\d+', s):
            id_numeric.append(str(int(s)))
    return sorted(set(id_numeric)), sorted(set(code_values))


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

dl = connect(
    to='DATALAKE',
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
dl._init_connection()

cdwh_connection = connect(
    to='CDWH',
    user_params={
        'user_name': khd_user,
        'password': khd_password,
    }
)
cdwh_connection._init_connection()

print('Impala + Datalake + CDWH initialized')

In [ ]:
# 1) Активные терминалы по месяцам периода + first_d_ter_delivery по serial
month_rows_sql = []
for m in period_months:
    month_start = m.strftime('%Y-%m-%d')
    month_end = (m + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    month_rows_sql.append(
        f"select cast('{month_start}' as date) as snapshot_month_start, cast('{month_end}' as date) as snapshot_month_end"
    )
months_sql = '\nunion all\n'.join(month_rows_sql)

sql_terminal_months = f"""
with months as (
{months_sql}
),
base as (
    select
      cast(t.c_nter as string) as c_nter,
      cast(t.c_pos_serial as string) as c_pos_serial,
      cast(t.d_ter_install as date) as d_ter_install,
      cast(t.d_ter_close as date) as d_ter_close,
      cast(t.d_ter_delivery as date) as d_ter_delivery
    from ods_alpha.scd1_pos_terminals t
    where t.c_nter is not null
      and t.c_pos_serial is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
      and coalesce(cast(t.d_ter_install as date), cast('1900-01-01' as date)) < cast('{period_end_exclusive}' as date)
      and coalesce(cast(t.d_ter_close as date), cast('2999-12-31' as date)) >= cast('{period_start}' as date)
),
first_deliver as (
    select
      cast(t.c_pos_serial as string) as c_pos_serial,
      min(cast(t.d_ter_delivery as date)) as first_d_ter_delivery
    from ods_alpha.scd1_pos_terminals t
    where t.c_pos_serial is not null
      and t.d_ter_delivery is not null
      and coalesce(t.ods_deleted_flg, '0') <> '1'
    group by cast(t.c_pos_serial as string)
),
active_raw as (
    select
      cast(m.snapshot_month_start as date) as snapshot_month_start,
      cast(b.c_nter as string) as c_nter,
      cast(b.c_pos_serial as string) as c_pos_serial,
      cast(fd.first_d_ter_delivery as date) as first_d_ter_delivery,
      cast(b.d_ter_install as date) as d_ter_install,
      cast(b.d_ter_close as date) as d_ter_close,
      row_number() over (
        partition by cast(m.snapshot_month_start as date), cast(b.c_nter as string)
        order by
          coalesce(cast(b.d_ter_install as date), cast('1900-01-01' as date)) desc,
          coalesce(cast(b.d_ter_close as date), cast('2999-12-31' as date)) desc,
          cast(b.c_pos_serial as string) desc
      ) as rn
    from months m
    join base b
      on coalesce(cast(b.d_ter_install as date), cast('1900-01-01' as date)) <= m.snapshot_month_end
     and coalesce(cast(b.d_ter_close as date), cast('2999-12-31' as date)) >= m.snapshot_month_start
    left join first_deliver fd
      on fd.c_pos_serial = b.c_pos_serial
)
select
  cast(snapshot_month_start as string) as snapshot_month_start,
  cast(c_nter as string) as c_nter,
  cast(c_pos_serial as string) as c_pos_serial,
  cast(first_d_ter_delivery as date) as first_d_ter_delivery,
  cast(d_ter_install as date) as d_ter_install,
  cast(d_ter_close as date) as d_ter_close
from active_raw
where rn = 1
"""

with imp:
    imp.execute('set MEM_LIMIT=16g')
    terminals_monthly_df = imp.fetch(sql_terminal_months)

if terminals_monthly_df is None:
    terminals_monthly_df = pd.DataFrame(
        columns=['snapshot_month_start', 'c_nter', 'c_pos_serial', 'first_d_ter_delivery', 'd_ter_install', 'd_ter_close']
    )

for c in ['snapshot_month_start', 'c_nter', 'c_pos_serial']:
    if c in terminals_monthly_df.columns:
        terminals_monthly_df[c] = terminals_monthly_df[c].astype(str).str.strip()

terminals_monthly_df['nter_norm'] = terminals_monthly_df['c_nter'].apply(norm_terminal_key)

print('terminals_monthly rows =', len(terminals_monthly_df))
print('distinct c_nter =', terminals_monthly_df['c_nter'].nunique() if len(terminals_monthly_df) else 0)
display(terminals_monthly_df.head(10))

In [ ]:
# 2) Модели терминалов из CDWH (Oracle-safe batching)
nter_values = clean_keys(terminals_monthly_df['c_nter'].tolist()) if len(terminals_monthly_df) else []
id_values_numeric, code_values = split_nter_for_id_and_code(nter_values)

cdwh_id_batches = 0
cdwh_code_batches = 0
cdwh_code_fallback_batches = 0
bm_chunks = []

if nter_values:
    with cdwh_connection:
        if id_values_numeric:
            for id_chunk in iter_chunks(id_values_numeric, 950):
                id_in = ', '.join(id_chunk)
                sql_by_id = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.ID_DEVICE in ({id_in})
                """
                part_df = cdwh_connection.fetch(sql_by_id)
                cdwh_id_batches += 1
                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

        if code_values:
            for code_chunk in iter_chunks(code_values, 900):
                code_in = sql_in(code_chunk)
                sql_by_code_fast = f"""
                select distinct
                  trim(to_char(d.ID_DEVICE)) as device,
                  trim(to_char(d.CODE_DEVICE)) as code_device,
                  trim(to_char(d.MODEL_DEVICE)) as model_device
                from BMRT.BM_DET_DEVICE d
                where d.CODE_DEVICE in ({code_in})
                """
                try:
                    part_df = cdwh_connection.fetch(sql_by_code_fast)
                    cdwh_code_batches += 1
                except Exception as exc:
                    sql_by_code_fallback = f"""
                    select distinct
                      trim(to_char(d.ID_DEVICE)) as device,
                      trim(to_char(d.CODE_DEVICE)) as code_device,
                      trim(to_char(d.MODEL_DEVICE)) as model_device
                    from BMRT.BM_DET_DEVICE d
                    where trim(to_char(d.CODE_DEVICE)) in ({code_in})
                    """
                    part_df = cdwh_connection.fetch(sql_by_code_fallback)
                    cdwh_code_fallback_batches += 1
                    print(f'CODE_DEVICE fallback chunk due to: {type(exc).__name__}')

                if part_df is not None and len(part_df):
                    bm_chunks.append(part_df)

if bm_chunks:
    bm_det_device_df = pd.concat(bm_chunks, ignore_index=True)
else:
    bm_det_device_df = pd.DataFrame(columns=['device', 'code_device', 'model_device'])

bm_det_device_df.columns = [str(c).strip().lower() for c in bm_det_device_df.columns]
for col in ['device', 'code_device', 'model_device']:
    if col not in bm_det_device_df.columns:
        bm_det_device_df[col] = np.nan
    bm_det_device_df[col] = bm_det_device_df[col].map(clean_cdwh_text)

bm_det_device_df['device_norm'] = bm_det_device_df['device'].apply(norm_terminal_key)
bm_det_device_df['code_device_norm'] = bm_det_device_df['code_device'].apply(norm_terminal_key)
_before_bm = len(bm_det_device_df)
bm_det_device_df = bm_det_device_df.loc[bm_det_device_df['model_device'].notna()].copy()
print(f'bm_det_device: dropped empty MODEL_DEVICE rows: {_before_bm - len(bm_det_device_df):,}')
bm_det_device_df = bm_det_device_df.drop_duplicates(subset=['device', 'code_device', 'model_device'])

print('CDWH batch stats:')
print('  by_id =', cdwh_id_batches)
print('  by_code_fast =', cdwh_code_batches)
print('  by_code_fallback =', cdwh_code_fallback_batches)
print('bm_det_device rows =', len(bm_det_device_df))
display(bm_det_device_df.head(10))


In [ ]:
# 3) Цены моделей из Excel + join к terminal-month perimeter
price_src_df = pd.read_excel(term_model_excel_path)
required_price_cols = ['model_name', 'price']
missing_price_cols = [c for c in required_price_cols if c not in price_src_df.columns]
if missing_price_cols:
    raise RuntimeError(f'В term_model.xlsx отсутствуют колонки: {missing_price_cols}')

price_map_df = price_src_df[['model_name', 'price']].copy()
price_map_df['model_key'] = price_map_df['model_name'].apply(norm_model)
price_map_df['price'] = pd.to_numeric(price_map_df['price'], errors='coerce')
price_map_df = (
    price_map_df.dropna(subset=['model_key', 'price'])
    .groupby('model_key', as_index=False)
    .agg(price=('price', 'max'))
)

# Model mapping priority: ID_DEVICE first, then CODE_DEVICE
if len(bm_det_device_df):
    device_map_df = (
        bm_det_device_df.loc[
            bm_det_device_df['device_norm'].notna()
            & bm_det_device_df['model_device'].notna(),
            ['device_norm', 'model_device']
        ]
        .drop_duplicates(subset=['device_norm'], keep='first')
    )
    code_map_df = (
        bm_det_device_df.loc[
            bm_det_device_df['code_device_norm'].notna()
            & bm_det_device_df['model_device'].notna(),
            ['code_device_norm', 'model_device']
        ]
        .drop_duplicates(subset=['code_device_norm'], keep='first')
    )
    device_map = dict(zip(device_map_df['device_norm'], device_map_df['model_device']))
    code_map = dict(zip(code_map_df['code_device_norm'], code_map_df['model_device']))
else:
    device_map = {}
    code_map = {}

amort_base_df = terminals_monthly_df.copy()
amort_base_df['model_device'] = amort_base_df['nter_norm'].map(device_map)
amort_base_df['join_key_used'] = np.where(
    amort_base_df['model_device'].notna(), 'id_device', None
)

need_code_mask = amort_base_df['model_device'].isna()
amort_base_df.loc[need_code_mask, 'model_device'] = amort_base_df.loc[need_code_mask, 'nter_norm'].map(code_map)
amort_base_df.loc[need_code_mask & amort_base_df['model_device'].notna(), 'join_key_used'] = 'code_device'

# sanitize again (never keep literal 'None' / 'nan')
amort_base_df['model_device'] = amort_base_df['model_device'].map(clean_cdwh_text)
amort_base_df.loc[amort_base_df['model_device'].isna(), 'join_key_used'] = None

amort_base_df['model_key'] = amort_base_df['model_device'].apply(norm_model)
amort_base_df.loc[amort_base_df['model_key'].isna(), ['model_device', 'join_key_used']] = np.nan

amort_base_df = amort_base_df.merge(price_map_df, on='model_key', how='left')

print('price_map rows =', len(price_map_df))
print('amort_base rows =', len(amort_base_df))
print(
    'model found =', int(amort_base_df['model_device'].notna().sum()),
    '| price found =', int(pd.to_numeric(amort_base_df['price'], errors='coerce').notna().sum()),
)
display(amort_base_df.head(10))


In [ ]:
# 4) Расчет амортизации по месяцам
amort_df = amort_base_df.copy()

amort_df['snapshot_month_start'] = pd.to_datetime(amort_df['snapshot_month_start'], errors='coerce')
amort_df['first_d_ter_delivery'] = pd.to_datetime(amort_df['first_d_ter_delivery'], errors='coerce')

amort_df['missing_serial'] = is_blank_series(amort_df['c_pos_serial'])
amort_df['missing_deliver'] = amort_df['first_d_ter_delivery'].isna()
# missing_model: no usable CDWH model (null / literal None / bad model_key)
amort_df['missing_model'] = (
    is_blank_series(amort_df['model_device'])
    | amort_df['model_key'].isna()
)
# missing_price: model known, but no Excel price (true CDWH→Excel gap)
amort_df['missing_price'] = (~amort_df['missing_model']) & pd.to_numeric(amort_df['price'], errors='coerce').isna()


def qc_status_fn(row):
    if row['missing_serial']:
        return 'missing_serial'
    if row['missing_deliver']:
        return 'missing_deliver'
    if row['missing_model']:
        return 'missing_model'
    if row['missing_price']:
        return 'missing_price'
    return 'complete'


amort_df['qc_status'] = amort_df.apply(qc_status_fn, axis=1)

complete_mask = amort_df['qc_status'] == 'complete'

first_month_first_day = amort_df.loc[complete_mask, 'first_d_ter_delivery'].dt.to_period('M').dt.to_timestamp()
report_month_first_day = amort_df.loc[complete_mask, 'snapshot_month_start'].dt.to_period('M').dt.to_timestamp()

months_from_start = (
    (report_month_first_day.dt.year - first_month_first_day.dt.year) * 12
    + (report_month_first_day.dt.month - first_month_first_day.dt.month)
)

amort_df['months_from_start'] = np.nan
amort_df.loc[complete_mask, 'months_from_start'] = months_from_start

amort_df['is_in_48m_window'] = (
    amort_df['months_from_start'].notna()
    & (amort_df['months_from_start'] >= 0)
    & (amort_df['months_from_start'] < 48)
)

amort_df['amortization_monthly'] = pd.to_numeric(amort_df['price'], errors='coerce') / 48.0
amort_df['amortization_for_report_month'] = amort_df['amortization_monthly'].where(amort_df['is_in_48m_window'], 0.0)
amort_df['amortization_for_report_month'] = amort_df['amortization_for_report_month'].fillna(0.0)

amort_df['report_month'] = amort_df['snapshot_month_start'].dt.strftime('%Y-%m')
amort_df['snapshot_month_start'] = amort_df['snapshot_month_start'].dt.strftime('%Y-%m-%d')
amort_df['first_d_ter_delivery'] = amort_df['first_d_ter_delivery'].dt.strftime('%Y-%m-%d')

amort_df['is_in_48m_window'] = amort_df['is_in_48m_window'].astype(int)
amort_df['is_amortized'] = (amort_df['amortization_for_report_month'] > 0).astype(int)

# --- Exclude brands/models from amortization (business rule) ---
# Zero amort for brands: Tactilion, VeriFone, Ingenico, IRAS (all models).
# Zero amort for Pax models: s80, s90, D210 (name variants).
# Delivery-date / 48m window logic for other models is unchanged.

import re as _re_excl

_EXCLUDE_BRAND_RE = _re_excl.compile(
    r'(tactilion|verifone|veri\s*fone|ingenico|iras)',
    _re_excl.IGNORECASE,
)
# Pax family restricted to s80 / s90 / d210 (allow separators/suffixes)
_EXCLUDE_PAX_RE = _re_excl.compile(
    r'(?:^|[^a-z0-9])pax(?:[^a-z0-9]+|\s+).*(?:s\s*80|s\s*90|d\s*210)'
    r'|(?:^|[^a-z0-9])(?:s\s*80|s\s*90|d\s*210).*(?:^|[^a-z0-9])pax'
    r'|(?:^|[^a-z0-9])pax\s*(?:s\s*80|s\s*90|d\s*210)',
    _re_excl.IGNORECASE,
)


def _exclude_amort_by_model(model_device, model_key=None):
    text = ' '.join(
        str(x) for x in (model_device, model_key)
        if x is not None and str(x).strip() not in ('', 'None', 'nan', 'NaN')
    )
    if not text.strip():
        return False
    if _EXCLUDE_BRAND_RE.search(text):
        return True
    if _EXCLUDE_PAX_RE.search(text):
        return True
    # also catch model strings that are clearly Pax s80/s90/d210 without brand token order issues
    t = text.lower().replace('\xa0', ' ')
    if 'pax' in t and any(tok in t.replace(' ', '') for tok in ('s80', 's90', 'd210')):
        return True
    return False


amort_df['exclude_amort_by_model'] = [
    _exclude_amort_by_model(md, mk)
    for md, mk in zip(amort_df.get('model_device'), amort_df.get('model_key'))
]

_amort_before_exclude = float(
    pd.to_numeric(amort_df['amortization_for_report_month'], errors='coerce').fillna(0).sum()
)
_excl_mask = amort_df['exclude_amort_by_model'].astype(bool)
amort_df.loc[_excl_mask, 'amortization_monthly'] = 0.0
amort_df.loc[_excl_mask, 'amortization_for_report_month'] = 0.0
amort_df['is_amortized'] = (pd.to_numeric(amort_df['amortization_for_report_month'], errors='coerce').fillna(0) > 0).astype(int)

_amort_after_exclude = float(
    pd.to_numeric(amort_df['amortization_for_report_month'], errors='coerce').fillna(0).sum()
)
print(
    'exclude_amort_by_model rows =', int(_excl_mask.sum()),
    '| amort_sum before =', round(_amort_before_exclude, 2),
    '| after =', round(_amort_after_exclude, 2),
    '| dropped =', round(_amort_before_exclude - _amort_after_exclude, 2),
)
_excl_top = (
    amort_df.loc[_excl_mask]
    .assign(model_device=amort_df.loc[_excl_mask, 'model_device'].astype(str))
    .groupby(['model_device'], as_index=False)
    .size()
    .sort_values('size', ascending=False)
    .head(30)
)
amort_df['exclude_amort_by_model'] = amort_df['exclude_amort_by_model'].astype(int)
print('=== Top excluded models (by row count) ===')
display(_excl_top)

amort_df['load_dt'] = pd.Timestamp.now().strftime('%Y-%m-%d')
amort_df['source_file'] = source_file

qc_missing_df = (
    amort_df.groupby('qc_status', as_index=False)
    .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
    .sort_values(['rows', 'terminals_nunique'], ascending=False)
    .reset_index(drop=True)
)

join_cov_df = amort_df.copy()
join_cov_df['join_key_used'] = join_cov_df['join_key_used'].fillna('no_match')
join_coverage_df = (
    join_cov_df.groupby('join_key_used', as_index=False)
    .agg(rows=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
    .sort_values(['rows', 'terminals_nunique'], ascending=False)
    .reset_index(drop=True)
)

print('QC: missing reasons')
display(qc_missing_df)
print('QC: join coverage')
display(join_coverage_df)
print('amort_df rows =', len(amort_df))
print('amortized rows =', int((amort_df['is_amortized'] == 1).sum()))
display(amort_df.head(10))


## 2. Проверка логичности (до загрузки)

Проверяем локальный `amort_df` / будущий `final_load_df` **до** записи в озеро:
- покрытие месяцев периода;
- нет дублей ключа `(snapshot_month_start, c_nter)`;
- суммы амортизации, доля exclude, qc_status;
- июль 2026 есть в снимке.

Если проверка падает — загрузку не запускаем.


In [ ]:
# 2) Pre-load sanity on local amort_df (before Impala write)
from IPython.display import display

expected_months = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']

if 'amort_df' not in globals() or amort_df is None or len(amort_df) == 0:
    raise RuntimeError('amort_df пуст — сначала выполните секцию расчёта')

pre = amort_df.copy()
pre['report_month'] = pre['report_month'].astype(str).str[:7]
pre['snapshot_month_start'] = pd.to_datetime(pre['snapshot_month_start'], errors='coerce')

# duplicates
dup_check = (
    pre.groupby(['snapshot_month_start', 'c_nter'], as_index=False)
    .size()
    .rename(columns={'size': 'cnt'})
)
dup_cnt = int((dup_check['cnt'] > 1).sum()) if len(dup_check) else 0
print('duplicate keys (snapshot_month_start, c_nter) =', dup_cnt)
if dup_cnt:
    display(dup_check.loc[dup_check['cnt'] > 1].head(20))
    raise RuntimeError('Дубли по ключу — загрузка запрещена')

# monthly rollup
pre['_amort'] = pd.to_numeric(pre['amortization_for_report_month'], errors='coerce').fillna(0)
if 'exclude_amort_by_model' in pre.columns:
    pre['_excl'] = pd.to_numeric(pre['exclude_amort_by_model'], errors='coerce').fillna(0).astype(int)
else:
    pre['_excl'] = 0
monthly_pre = (
    pre.groupby('report_month', as_index=False)
    .agg(
        rows_cnt=('c_nter', 'size'),
        terminals_nunique=('c_nter', 'nunique'),
        amortization_total=('_amort', 'sum'),
        exclude_rows=('_excl', 'sum'),
    )
    .sort_values('report_month')
)

print('=== Pre-load monthly summary (local amort_df) ===')
display(monthly_pre)

have_months = set(monthly_pre['report_month'].astype(str).str[:7].tolist())
missing = [m for m in expected_months if m not in have_months]
extra = sorted(have_months - set(expected_months))
print('missing months:', missing)
print('extra months:', extra)
if missing:
    raise RuntimeError(f'В локальном расчёте нет месяцев: {missing}')

# qc_status breakdown
if 'qc_status' in pre.columns:
    qc_pre = (
        pre.groupby('qc_status', dropna=False, as_index=False)
        .agg(rows_cnt=('c_nter', 'size'), terminals_nunique=('c_nter', 'nunique'))
        .sort_values('rows_cnt', ascending=False)
    )
    print('=== qc_status ===')
    display(qc_pre)

amort_total = float(pd.to_numeric(pre['amortization_for_report_month'], errors='coerce').fillna(0).sum())
excl_rows = int(pd.to_numeric(pre.get('exclude_amort_by_model'), errors='coerce').fillna(0).astype(int).sum()) if 'exclude_amort_by_model' in pre.columns else 0
amort_pos = int((pd.to_numeric(pre['amortization_for_report_month'], errors='coerce').fillna(0) > 0).sum())
print(
    'rows =', len(pre),
    '| amort_total =', round(amort_total, 2),
    '| amort>0 rows =', amort_pos,
    '| exclude_amort_by_model rows =', excl_rows,
)

july = pre.loc[pre['report_month'] == '2026-07']
print('july rows =', len(july), '| july amort =', round(float(pd.to_numeric(july['amortization_for_report_month'], errors='coerce').fillna(0).sum()), 2))

print('OK: pre-load checks passed — можно загружать в озеро')


## 3. Загрузка в Impala / DRP (datalake)

Подготовка `final_load_df` и полная перезапись
`sandbox_ai.shestopalov_terminal_amortization_model` (DROP + CREATE + ORC).

Запускать только после успешной проверки в секции 2.


In [ ]:
# 5) Итоговый датафрейм для загрузки в target table
load_cols = [
    'snapshot_month_start',
    'report_month',
    'c_nter',
    'c_pos_serial',
    'first_d_ter_delivery',
    'model_device',
    'model_key',
    'price',
    'amortization_monthly',
    'months_from_start',
    'is_in_48m_window',
    'amortization_for_report_month',
    'is_amortized',
    'exclude_amort_by_model',
    'join_key_used',
    'qc_status',
    'missing_serial',
    'missing_deliver',
    'missing_model',
    'missing_price',
    'load_dt',
    'source_file',
]

for c in load_cols:
    if c not in amort_df.columns:
        amort_df[c] = None

final_load_df = amort_df[load_cols].copy()

# Однозначность ключа внутри периода
dup_check = (
    final_load_df.groupby(['snapshot_month_start', 'c_nter'], as_index=False)
    .size()
    .rename(columns={'size': 'cnt'})
)
dup_cnt = int((dup_check['cnt'] > 1).sum()) if len(dup_check) else 0
print('duplicate keys (snapshot_month_start, c_nter) =', dup_cnt)
if dup_cnt:
    display(dup_check[dup_check['cnt'] > 1].head(20))
    raise RuntimeError('Найдены дубли по ключу (snapshot_month_start, c_nter). Загрузка остановлена.')

print('final_load_df rows =', len(final_load_df))
print('final_load_df months =', sorted(final_load_df['report_month'].dropna().astype(str).unique().tolist()))
display(final_load_df.head(20))


In [ ]:
# 6) Полная перезапись target table в Datalake/Impala (DROP + CREATE + ORC load)

def load_to_datalake_from_file(local_file_path: str, table: str, dest_catalog: str = 'external', cleanup_before_copy: bool = True):
    schema = table.split('.')[0]
    table_name = table.split('.')[-1]
    hdfs_path = f'/warehouse/tablespace/{dest_catalog}/hive/{schema}.db/{table_name}'

    if cleanup_before_copy:
        # Удаляем старые файлы, чтобы избежать дублей после полной перезагрузки
        subprocess.run(['hdfs', 'dfs', '-rm', '-r', '-f', f'{hdfs_path}/*'], check=False)

    subprocess.run(['hdfs', 'dfs', '-copyFromLocal', '-f', local_file_path, f'{hdfs_path}/{local_file_path}'], check=True)
    out = subprocess.run(['hdfs', 'dfs', '-ls', '-h', hdfs_path], capture_output=True, text=True, check=True)
    print(f'Files in HDFS path {hdfs_path}:\n{out.stdout}')


# Подготовка типов и null для записи
load_df = final_load_df.copy()
for c in ['price', 'amortization_monthly', 'months_from_start', 'amortization_for_report_month']:
    load_df[c] = pd.to_numeric(load_df[c], errors='coerce')
for c in ['is_in_48m_window', 'is_amortized', 'exclude_amort_by_model', 'missing_serial', 'missing_deliver', 'missing_model', 'missing_price']:
    load_df[c] = pd.to_numeric(load_df[c], errors='coerce').fillna(0).astype('int64')

load_df = load_df.fillna({
    'snapshot_month_start': '',
    'report_month': '',
    'c_nter': '',
    'c_pos_serial': '',
    'first_d_ter_delivery': '',
    'model_device': '',
    'model_key': '',
    'join_key_used': '',
    'qc_status': '',
    'load_dt': '',
    'source_file': source_file,
})

load_df.to_orc(save_table_orc_name, index=False)
print('ORC prepared:', save_table_orc_name, 'rows=', len(load_df))

create_sql = f"""
create external table if not exists {target_table} (
    snapshot_month_start string,
    report_month string,
    c_nter string,
    c_pos_serial string,
    first_d_ter_delivery string,
    model_device string,
    model_key string,
    price double,
    amortization_monthly double,
    months_from_start double,
    is_in_48m_window bigint,
    amortization_for_report_month double,
    is_amortized bigint,
    exclude_amort_by_model bigint,
    join_key_used string,
    qc_status string,
    missing_serial bigint,
    missing_deliver bigint,
    missing_model bigint,
    missing_price bigint,
    load_dt string,
    source_file string
)
stored as orc
 tblproperties ('transactional'='false')
"""

with dl:
    dl.execute(f'drop table if exists {target_table}')
    dl.execute(create_sql)

load_to_datalake_from_file(save_table_orc_name, target_table, dest_catalog='external', cleanup_before_copy=True)

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')

print('Reload completed:', target_table)


## 4. Smoke после загрузки

Короткий контроль, что таблица в озере реально перезаписалась
(invalidate/refresh + месяцы + июль). Полная логика уже проверена до load.


In [ ]:
# 4) Post-load smoke: таблица в озере перезаписалась
from IPython.display import display

expected_months = ['2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']

sql_monthly_smoke = f"""
select
  cast(report_month as string) as report_month,
  count(*) as rows_cnt,
  count(distinct cast(c_nter as string)) as terminals_nunique,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as amortization_total,
  sum(coalesce(cast(exclude_amort_by_model as bigint), 0)) as exclude_rows
from {target_table}
group by cast(report_month as string)
order by 1
"""

sql_july_smoke = f"""
select
  count(*) as july_rows,
  count(distinct cast(c_nter as string)) as july_terminals,
  sum(coalesce(cast(amortization_for_report_month as double), 0.0)) as july_amort
from {target_table}
where cast(report_month as string) = '2026-07'
   or cast(snapshot_month_start as string) = '2026-07-01'
"""

sql_dup = f"""
select count(*) as duplicated_month_nter_cnt
from (
  select snapshot_month_start, c_nter
  from {target_table}
  group by snapshot_month_start, c_nter
  having count(*) > 1
) d
"""

with imp:
    imp.execute(f'invalidate metadata {target_table}')
    imp.execute(f'refresh {target_table}')
    monthly_smoke_df = imp.fetch(sql_monthly_smoke)
    july_smoke_df = imp.fetch(sql_july_smoke)
    dup_smoke_df = imp.fetch(sql_dup)

print('Smoke table:', target_table)
display(monthly_smoke_df)
display(july_smoke_df)
display(dup_smoke_df)

have = set(monthly_smoke_df['report_month'].astype(str).str[:7].tolist()) if monthly_smoke_df is not None and len(monthly_smoke_df) else set()
missing = [m for m in expected_months if m not in have]
dup_cnt = int(pd.to_numeric(dup_smoke_df['duplicated_month_nter_cnt'], errors='coerce').fillna(0).iloc[0]) if dup_smoke_df is not None and len(dup_smoke_df) else -1
print('missing months in lake:', missing)
print('duplicated keys in lake:', dup_cnt)
if missing:
    raise RuntimeError(f'После load в озере нет месяцев: {missing}')
if dup_cnt != 0:
    raise RuntimeError(f'После load в озере есть дубли ключа: {dup_cnt}')

# optional: compare local pre-load amort_total vs lake
if 'amort_df' in globals() and amort_df is not None and len(amort_df):
    local_total = float(pd.to_numeric(amort_df['amortization_for_report_month'], errors='coerce').fillna(0).sum())
    lake_total = float(pd.to_numeric(monthly_smoke_df['amortization_total'], errors='coerce').fillna(0).sum()) if monthly_smoke_df is not None else float('nan')
    print('amort_total local =', round(local_total, 2), '| lake =', round(lake_total, 2), '| delta =', round(lake_total - local_total, 2))
    if abs(lake_total - local_total) > 1.0:
        raise RuntimeError('Сумма амортизации в озере расходится с локальным расчётом > 1 руб')

print('OK: post-load smoke passed')
